In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════════════════════
#  PHASE 2 — FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════
print("="*65)
print("  PHASE 2: FEATURE ENGINEERING")
print("="*65)

df = combined.copy()

# 7-day rolling mean & std per unit (chronological order)
df['rolling_mean_7'] = (
    df.groupby('unit_id')['daily_yield_kwh']
      .transform(lambda x: x.rolling(7, min_periods=2).mean())
)
df['rolling_std_7'] = (
    df.groupby('unit_id')['daily_yield_kwh']
      .transform(lambda x: x.rolling(7, min_periods=2).std().fillna(0))
)

# Z-score: how unusual is today vs this unit's own entire history
df['z_score'] = (
    df.groupby('unit_id')['daily_yield_kwh']
      .transform(lambda x: (x - x.mean()) / (x.std() + 1e-9))
)

# Fleet average: mean across ALL units on the same calendar date
fleet_avg = (
    df.groupby('date', as_index=False)
      .agg(fleet_avg_yield=('daily_yield_kwh', 'mean'))
)
df = df.merge(fleet_avg, on='date', how='left')

# Percent deviation from the fleet average that day
df['pct_dev_from_fleet'] = (
    (df['daily_yield_kwh'] - df['fleet_avg_yield'])
    / (df['fleet_avg_yield'] + 1e-9)
) * 100

# Day-on-day percentage change
df['day_change_pct'] = (
    df.groupby('unit_id')['daily_yield_kwh']
      .pct_change().fillna(0) * 100
)

# Lag features
df['lag_1'] = df.groupby('unit_id')['daily_yield_kwh'].shift(1)
df['lag_7'] = df.groupby('unit_id')['daily_yield_kwh'].shift(7)

# Ratio of today's output to the recent 7-day rolling mean
df['ratio_to_rollmean'] = (
    df['daily_yield_kwh'] / (df['rolling_mean_7'] + 1e-9)
)

# Performance ratio: daily yield per kW of installed capacity
df['performance_ratio'] = (
    df['daily_yield_kwh'] / (df['installed_kw'] + 1e-9)
)

# Drop rows that still lack enough rolling/lag history
feature_df = df.dropna(subset=['rolling_mean_7', 'lag_1']).copy().reset_index(drop=True)
print(f"✅ Features created | {len(feature_df)} rows | {feature_df['unit_id'].nunique()} units")
